In [1]:
import pandas as pd
import numpy as np

In [2]:
PATH = './data'
TARGET = 'SalePrice'

In [3]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

In [4]:
df = pd.read_csv(PATH + "/train.csv")

In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, shuffle=True, random_state=42
)

train_ids = X_train["Id"]
test_ids = X_test["Id"]

print(f"X_train shape {X_train.shape}")
print(f"X_test shape {X_test.shape}")

X_train shape (1168, 80)
X_test shape (292, 80)


### Data Cleaning ###

### Ordinal Columns ###

In [6]:
# Some categorical columns can may be ordered, replace them with numbers.

cat_ord_cols = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'HeatingQC',
    'KitchenQual',
    'Functional',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

exter_qual_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
exter_cond_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_qual_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_cond_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_exposure_order     = ['NA', 'No', 'Mn', 'Av', 'Gd']
bsmt_fin_type_1_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
bsmt_fin_type_2_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
heating_qc_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
kitchen_qual_order      = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
functional_order        = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
fireplace_qu_order      = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_finish_order     = ['NA', 'Unf', 'RFn', 'Fin']
garage_qual_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_cond_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
pool_qc_order           = ['NA', 'Fa', 'TA', 'Gd', 'Ex']
fence_order             = ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']

ord_categories = [
    exter_qual_order,
    exter_cond_order,
    bsmt_qual_order,
    bsmt_cond_order,
    bsmt_exposure_order,
    bsmt_fin_type_1_order,
    bsmt_fin_type_2_order,
    heating_qc_order,
    kitchen_qual_order,
    functional_order,
    fireplace_qu_order,
    garage_finish_order,
    garage_qual_order,
    garage_cond_order,
    pool_qc_order,
    fence_order,
]

### Categorical Imputing ###

In [103]:
cat_impute_cols = [
    'ExterQual',
    'ExterCond',
    'HeatingQC',
    'KitchenQual',
    'Functional',
]

### Categorical One-Hot-Encoded Columns ###

In [7]:
# Here we list features to be one-hot-encoded including 'NaN's because 'NaN' here does not mean missing.
# 'MasVnrType' was interesting column here.

cat_ohe_cols = [
    'MSZoning',
    'Street',
    'Alley',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'MasVnrType',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'GarageType',
    'PavedDrive',
    'MiscFeature',
    'SaleType',
    'SaleCondition',
]

### Numeric One-Hot-Encoded Columns ###

In [8]:
# This feature has numeric type which is misleading.

num_ohe_cols = [
    'MSSubClass',
]

### Numeric Imputing ###

In [9]:
num_nan_cols = [
    'LotFrontage',
    'GarageYrBlt',
    'MasVnrArea',
]

### Data Cleanup Pipeline ###

In [100]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

ohe_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='Missing')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='NA')),
    ('ordinal', OrdinalEncoder(categories=ord_categories)),
])

num_ohe_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', missing_values=np.nan, fill_value=-1)),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('ord',     ordinal_pipeline,               cat_ord_cols),
        ('cat_ohe', ohe_pipeline,                   cat_ohe_cols),
        ('num_ohe', num_ohe_pipeline,               num_ohe_cols),
        ('num_nan', SimpleImputer(strategy='mean'), num_nan_cols),
        ('drop',    'drop',                         ['Id'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

preprocessor.set_output(transform='pandas')

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ord', ...), ('cat_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featur

In [97]:
# from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.feature_selection import RFE
#
# class CorrelationFilter(BaseEstimator, TransformerMixin):
#     def __init__(self, threshold=.5):
#         pass
#
#     def fit(self, X, y):
#         pass
#
#     def transform(self, X):
#         pass

In [101]:
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler

preprocessor_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    # ('scaler', StandardScaler())
    # ('rfe', RFE(estimator=LinearRegression(), n_features_to_select=10))
])

preprocessor_pipeline.set_output(transform='pandas')

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ord', ...), ('cat_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [90]:
preprocessor_pipeline.fit_transform(X_train, y_train)
X_train_t = preprocessor_pipeline.transform(X_train)
X_test_t = preprocessor_pipeline.transform(X_test)
y_train_t = np.log1p(y_train)
y_test_t = np.log1p(y_test)

print(X_train_t.shape)
print(X_test_t.shape)
print(y_train_t.shape)
print(y_test_t.shape)

(1168, 233)
(292, 233)
(1168,)
(292,)


### Full Pipeline ###

In [28]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Ridge())
])

param_grid = {
    'preprocessor__preprocessor__num_nan__strategy' : ['mean', 'median'],
    'model__alpha': [.01, .05, .1, .5, 1, 5, 15, 20, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500],
}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)

cv_results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,param_preprocessor__preprocessor__num_nan__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.193223,0.024906,0.097852,0.022976,0.01,mean,"{'model__alpha': 0.01, 'preprocessor__preproce...",-0.136832,-0.161168,-0.203389,-0.132375,-0.160986,-0.158950,0.025216,39,-0.087774,-0.087264,-0.091625,-0.092641,-0.088938,-0.089648,0.002124
1,0.117164,0.014031,0.064819,0.005593,0.01,median,"{'model__alpha': 0.01, 'preprocessor__preproce...",-0.136805,-0.161136,-0.203439,-0.132410,-0.161028,-0.158963,0.025231,40,-0.087763,-0.087261,-0.091611,-0.092623,-0.088919,-0.089635,0.002121
2,0.119000,0.047874,0.055145,0.006425,0.05,mean,"{'model__alpha': 0.05, 'preprocessor__preproce...",-0.136816,-0.161131,-0.203361,-0.132340,-0.160926,-0.158915,0.025215,37,-0.087774,-0.087264,-0.091625,-0.092641,-0.088938,-0.089648,0.002124
3,0.117678,0.023290,0.064645,0.014577,0.05,median,"{'model__alpha': 0.05, 'preprocessor__preproce...",-0.136789,-0.161099,-0.203410,-0.132375,-0.160968,-0.158928,0.025230,38,-0.087763,-0.087261,-0.091611,-0.092623,-0.088919,-0.089635,0.002121
4,0.222774,0.093355,0.070907,0.011709,0.10,mean,"{'model__alpha': 0.1, 'preprocessor__preproces...",-0.136796,-0.161084,-0.203326,-0.132297,-0.160852,-0.158871,0.025213,35,-0.087774,-0.087264,-0.091625,-0.092641,-0.088938,-0.089649,0.002124
5,0.144314,0.040584,0.066938,0.007458,0.10,median,"{'model__alpha': 0.1, 'preprocessor__preproces...",-0.136770,-0.161053,-0.203375,-0.132331,-0.160894,-0.158884,0.025228,36,-0.087763,-0.087261,-0.091611,-0.092623,-0.088919,-0.089635,0.002121
6,0.105501,0.008096,0.057847,0.002607,0.50,mean,"{'model__alpha': 0.5, 'preprocessor__preproces...",-0.136645,-0.160727,-0.203069,-0.131959,-0.160281,-0.158536,0.025206,33,-0.087775,-0.087265,-0.091626,-0.092642,-0.088940,-0.089650,0.002124
7,0.101720,0.009387,0.054291,0.001105,0.50,median,"{'model__alpha': 0.5, 'preprocessor__preproces...",-0.136618,-0.160695,-0.203120,-0.131992,-0.160324,-0.158550,0.025221,34,-0.087764,-0.087262,-0.091612,-0.092624,-0.088920,-0.089637,0.002121
8,0.108871,0.015047,0.054031,0.000970,1.00,mean,"{'model__alpha': 1, 'preprocessor__preprocesso...",-0.136468,-0.160305,-0.202798,-0.131559,-0.159625,-0.158151,0.025209,31,-0.087778,-0.087269,-0.091629,-0.092645,-0.088944,-0.089653,0.002124
9,0.104869,0.017067,0.052635,0.002271,1.00,median,"{'model__alpha': 1, 'preprocessor__preprocesso...",-0.136441,-0.160273,-0.202850,-0.131590,-0.159668,-0.158164,0.025226,32,-0.087767,-0.087266,-0.091615,-0.092627,-0.088924,-0.089640,0.002120


In [59]:
from sklearn.metrics import root_mean_squared_error

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Ridge())
])

final_pipeline.fit(X_train, y_train_t)
y_pred = final_pipeline.predict(X_test)
print(root_mean_squared_error(y_pred, y_test_t))

0.12305799643073716


In [102]:
final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Ridge())
])

test = pd.read_csv(PATH + '/test.csv')
test_ids = test['Id']

Xt = preprocessor_pipeline.fit_transform(test)
print(Xt.isna().sum().sort_values(ascending=False))

# test['LandContour'].value_counts(dropna=False)
# prices_log = final_pipeline.predict(test)
# prices = np.expm1(prices_log)
#
# submission = pd.DataFrame({
#     "Id": test_ids,
#     "SalePrice": prices,
# })
#
# submission.to_csv("submission.csv", index=False)

ValueError: Found unknown categories ['NA'] in column 8 during fit